In [228]:
import pandas as pd

In [229]:
df = pd.read_csv('../data/Titanic-Dataset.csv')

In [230]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [231]:
df.shape

(891, 12)

In [232]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


### Some reusable functions:

In [233]:
# Drop Columns
def drop_cols(cols, df):
    df.drop(columns=cols, inplace=True)

## Experiment #01
- Dropped irrelevant columns: `Name`, `PassengerId`, `Ticket`
- Handled categorical variable `Sex` using binary mapping (0/1)
- Applied One-Hot Encoding on `Embarked`
- Trained a Logistic Regression model for classification

In [234]:
df_exp1 = df.copy()

### Handling Missing Values

In [235]:
cols_with_missing_values = ['Age', 'Cabin', 'Embarked']
for col in cols_with_missing_values:
    missing_percent = (df_exp1[col].isnull().mean()) * 100
    print(f"{col} has {missing_percent}% missing values")

Age has 19.865319865319865% missing values
Cabin has 77.10437710437711% missing values
Embarked has 0.22446689113355783% missing values


In [236]:
# Cabin has 70+% data missing so dropping it will be fine
df_exp1.drop(columns='Cabin', inplace=True)
df.drop(columns='Cabin', inplace=True)

# Mean Imputation on Age
df_exp1['Age'] = df_exp1['Age'].fillna(df_exp1['Age'].mean())

# Mode Imputation on Age
df_exp1['Embarked'] = df_exp1['Embarked'].fillna(df_exp1['Embarked'].mode()[0])

df_exp1.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Embarked     891 non-null    str    
dtypes: float64(2), int64(5), str(4)
memory usage: 76.7 KB


In [237]:
df_exp1.drop(columns=['Name', 'PassengerId', 'Ticket'], inplace=True)

In [238]:
from sklearn.model_selection import train_test_split

X = df_exp1.drop("Survived", axis=1)
y = df_exp1["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### One Hot Encoding on 'Embarked'

In [239]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

X_train_encoded = encoder.fit_transform(X_train[['Embarked']])
X_test_encoded = encoder.transform(X_test[['Embarked']])

X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=encoder.get_feature_names_out(['Embarked']),
    index=X_train.index
)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=encoder.get_feature_names_out(['Embarked']),
    index=X_test.index
)

X_train = X_train.drop(columns=['Embarked'])
X_test = X_test.drop(columns=['Embarked'])

X_train = pd.concat([X_train, X_train_encoded], axis=1)
X_test = pd.concat([X_test, X_test_encoded], axis=1)

#### Encoding 'Sex' Feature

In [240]:
X_train["Sex"] = X_train["Sex"].map({
    "male": 0,
    "female": 1
})

X_test["Sex"] = X_test["Sex"].map({
    "male": 0,
    "female": 1
})

#### Logistic Regression

In [241]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8100558659217877

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.86      0.84       105
           1       0.79      0.74      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179


Confusion Matrix:
 [[90 15]
 [19 55]]


## Experiment #02 (Accuracy Improved)
- Added a missing indicator
- Everything else same as experiment 1

In [242]:
df_exp2 = df.copy()
df_exp2.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(4)
memory usage: 76.7 KB


In [243]:
df_exp2.drop(columns=['Name', 'PassengerId', 'Ticket'], inplace=True)

Handling missing values in 'Age' and 'Embarked' and keeping track of such records:

In [244]:
for col in ['Age', 'Embarked']:
    df_exp2['Missing' + col] = df_exp2[col].isnull().astype(int)

df_exp2['Age'] = df_exp2['Age'].fillna(df_exp2['Age'].mean())

df_exp2['Embarked'] = df_exp2['Embarked'].fillna(df_exp2['Embarked'].mode()[0])

df_exp2.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Survived         891 non-null    int64  
 1   Pclass           891 non-null    int64  
 2   Sex              891 non-null    str    
 3   Age              891 non-null    float64
 4   SibSp            891 non-null    int64  
 5   Parch            891 non-null    int64  
 6   Fare             891 non-null    float64
 7   Embarked         891 non-null    str    
 8   MissingAge       891 non-null    int64  
 9   MissingEmbarked  891 non-null    int64  
dtypes: float64(2), int64(6), str(2)
memory usage: 69.7 KB


In [245]:
X = df_exp2.drop("Survived", axis=1)
y = df_exp2["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### One Hot Encoding on 'Embarked'

In [246]:
encoder = OneHotEncoder(sparse_output=False)

X_train_encoded = encoder.fit_transform(X_train[['Embarked']])

X_test_encoded = encoder.transform(X_test[['Embarked']])

X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=encoder.get_feature_names_out(['Embarked']),
    index=X_train.index
)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=encoder.get_feature_names_out(['Embarked']),
    index=X_test.index
)

X_train = X_train.drop(columns=['Embarked'])
X_test = X_test.drop(columns=['Embarked'])

X_train = pd.concat([X_train, X_train_encoded], axis=1)
X_test = pd.concat([X_test, X_test_encoded], axis=1)

#### Converting 'Sex' to numerical feature

In [247]:
X_train["Sex"] = X_train["Sex"].map({
    "male": 0,
    "female": 1
})

X_test["Sex"] = X_test["Sex"].map({
    "male": 0,
    "female": 1
})

#### Logistic Regression

In [248]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8212290502793296

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.87      0.85       105
           1       0.80      0.76      0.78        74

    accuracy                           0.82       179
   macro avg       0.82      0.81      0.81       179
weighted avg       0.82      0.82      0.82       179


Confusion Matrix:
 [[91 14]
 [18 56]]


## Experiment #03 
- Applied Target Encoding 
- Everything else same as experiment 2

In [249]:
df_exp3 = df.copy()
df_exp3.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(4)
memory usage: 76.7 KB


In [250]:
df_exp3.drop(columns=['Name', 'PassengerId', 'Ticket'], inplace=True)

In [251]:
for col in ['Age', 'Embarked']:
    df_exp3['Missing' + col] = df_exp3[col].isnull().astype(int)

df_exp3['Age'] = df_exp3['Age'].fillna(df_exp3['Age'].mean())

df_exp3['Embarked'] = df_exp3['Embarked'].fillna(df_exp3['Embarked'].mode()[0])

df_exp3.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Survived         891 non-null    int64  
 1   Pclass           891 non-null    int64  
 2   Sex              891 non-null    str    
 3   Age              891 non-null    float64
 4   SibSp            891 non-null    int64  
 5   Parch            891 non-null    int64  
 6   Fare             891 non-null    float64
 7   Embarked         891 non-null    str    
 8   MissingAge       891 non-null    int64  
 9   MissingEmbarked  891 non-null    int64  
dtypes: float64(2), int64(6), str(2)
memory usage: 69.7 KB


In [252]:
X = df_exp3.drop("Survived", axis=1)
y = df_exp3["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Converting 'Sex' to numerical feature

In [253]:
X_train["Sex"] = X_train["Sex"].map({
    "male": 0,
    "female": 1
})

X_test["Sex"] = X_test["Sex"].map({
    "male": 0,
    "female": 1
})

#### Target Encoding on 'Embarked'

In [254]:
from sklearn.preprocessing import TargetEncoder

encoder = TargetEncoder(smooth=10.0) 

X_train['Embarked'] = encoder.fit_transform(X_train[['Embarked']], y_train)
X_test['Embarked'] = encoder.transform(X_test[['Embarked']])


#### Logistic Regression

In [255]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8044692737430168

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.86      0.84       105
           1       0.78      0.73      0.76        74

    accuracy                           0.80       179
   macro avg       0.80      0.79      0.80       179
weighted avg       0.80      0.80      0.80       179


Confusion Matrix:
 [[90 15]
 [20 54]]
